In [1]:
import numpy as np
from pulp import *
import math

# FUNCTIONS

In [2]:
np.random.seed(42) # for reproducibility

num_scenarios = 30
scenario = {s: {"kesi_1": np.random.randint(2, 10), "kesi_2": np.random.randint(2, 12)} for s in range(1, num_scenarios + 1)}

# Assume equal probability for each scenario
pscenario = [1.0 / num_scenarios] * num_scenarios

In [3]:
def check_cut_feasibility(x, master, h, T, scenario):
    ADD_CUT = False
    d=[]
    D=[]
    simplex_multiplier_feasibility = {}
    
    for s in range(1, num_scenarios + 1):
        kesi_1 = scenario[s]["kesi_1"]
        kesi_2 = scenario[s]["kesi_2"]

        w_prime=LpProblem('feasibility-check-problem' , LpMinimize)
        v=LpVariable.dicts('v' , ((i,j) for i in [1,2,3,4,5,6] for j in [1,2]), lowBound=0, cat=LpContinuous)
        y = LpVariable.dicts('y', [1,2], lowBound=0, cat=LpContinuous)
        w_prime += lpSum(v[i,j] for i in [1,2,3,4,5,6] for j in [1,2])
        w_prime += v[1,1] - v[1,2] + 3 * y[1] + 2 * y[2] -x[1].varValue <= 0 
        w_prime += v[2,1] - v[2,2] + 2 * y[1] + 5 * y[2] -x[2].varValue <= 0
        w_prime += v[3,1] - v[3,2] + y[1] >= 0.8 * kesi_1
        w_prime += v[4,1] - v[4,2] + y[2] >= 0.8 * kesi_2
        w_prime += v[5,1] - v[5,2] + y[1] <= kesi_1
        w_prime += v[6,1] - v[6,2] + y[2] <= kesi_2

        w_prime.solve(PULP_CBC_CMD(msg=False, keepFiles=False)) 
        # print(f" objective of w_prime problem is :    {value(w_prime.objective)}")    # h is extracted so just there is need to extract multipliers

        #extract multipliers of feasibility check lp
        sigma = []
        for _, c in w_prime.constraints.items():
            sigma.append(c.pi)
        simplex_multiplier_feasibility[s]= sigma

        d.append(np.array([simplex_multiplier_feasibility[s]]) @ np.array(h[s]))
        D.append(np.array([simplex_multiplier_feasibility[s]]) @ np.array(T[s]))
        # print(f" d is : {d}   ,   D is : {D}")

        if value(w_prime.objective) > 1e-8:
            # r += 1
            ADD_CUT = True
            d_val = d[-1][0]  
            D_vec = D[-1][0]  
            cut_expr = lpSum(D_vec[i] * x[i+1] for i in range(len(D_vec)))
            master += cut_expr >= d_val

            print(f"added feasibility cut to master problem :   {cut_expr >= d_val}")
    
    if ADD_CUT:
        return False
    return True

In [4]:
def check_cut_optimality(x, master, h, T, scenario, theta):
    e = []
    E = []
    ADD_CUT = False
    simplex_multiplier_optimaly = {}
    
    for s in range(1, num_scenarios + 1):
        kesi_1 = scenario[s]["kesi_1"]
        kesi_2 = scenario[s]["kesi_2"]
        
        sub = LpProblem('SubProblem', LpMinimize)
        y = LpVariable.dicts('y', [1,2], lowBound=0, cat=LpContinuous)
        sub += 15 * y[1] + 12 * y[2]
        sub += 3 * y[1] + 2 * y[2] - x[1].varValue <= 0
        sub += 2 * y[1] + 5 * y[2] - x[2].varValue <= 0
        sub += y[1] >= 0.8 *  kesi_1
        sub += y[1] <= kesi_1
        sub += y[2] >= 0.8 *  kesi_2
        sub += y[2] <= kesi_2


        sub.solve(PULP_CBC_CMD(msg=False ))

        pi = []
        for _, c in sub.constraints.items():
                        pi.append(-c.pi)
        simplex_multiplier_optimaly[s]= pi 
        # print(F"simplex multiplier  : {simplex_multiplier_optimaly}")

        e = pscenario[s-1] * np.array([simplex_multiplier_optimaly[s]]) @ np.array(h[s])        
        E = pscenario[s-1] * np.array([simplex_multiplier_optimaly[s]]) @ np.array(T[s])
        # print(f" e = {e} ,  E =  {E}")
        q = e - E @ np.array( [x[1].varValue , x[2].varValue] )

        # print(f" q  :   {q}")
        # print(f" theta s = {theta[s].varValue}")
        
        if q[0] > theta[s].varValue:

            ADD_CUT = True

            e_val = e[0]  
            E_vec = E[0]
            cut_expr_opt = lpSum(E_vec[i] * x[i+1] for i in range(len(E_vec))) + theta[s]
            master += cut_expr_opt >= e_val
            print(f"added optimaly cut to master problem :   {cut_expr_opt >= e_val}")

    if ADD_CUT:
        return False
    return True

# Main Code

In [5]:
r,s,k = (0,0,0)

master = LpProblem('masterproblem' , sense=LpMinimize)
x = LpVariable.dicts('x' , [1,2] , lowBound=0 , cat=LpContinuous )
theta = LpVariable.dicts('theta', range(1, num_scenarios + 1), lowBound=-100000,  cat=LpContinuous)

master += 3 * x[1] + 2 * x[2] + lpSum(theta[s] for s in range(1, num_scenarios + 1))
master += x[1] + x[2] <= 10000

In [7]:
#second stage and h , T extraction

h = {}
T = {}
for s in range(1, num_scenarios + 1):

    kesi_1 = scenario[s]["kesi_1"]
    kesi_2 = scenario[s]["kesi_2"]

    sub = LpProblem('SubProblem', LpMinimize)
    y = LpVariable.dicts('y', [1,2], lowBound=0, cat=LpContinuous)
    sub += 15 * y[1] + 12 * y[2]
    sub += 3 * y[1] + 2 * y[2] - x[1] <= 0
    sub += 2 * y[1] + 5 * y[2] - x[2] <= 0
    sub += y[1] >= 0.8 *  kesi_1
    sub += y[2] >= 0.8 *  kesi_2
    sub += y[1] <= kesi_1
    sub += y[2] <= kesi_2


    h_tmp = []
    for name, constraint in sub.constraints.items():
        h_tmp.append(-constraint.constant)
    h[s] = h_tmp
    
    T_tmp = []
    # It's better to iterate through the constraints in a fixed order
    for name in sorted(sub.constraints.keys()):
        constraint = sub.constraints[name]
        coeff_row = []
        # Ensure a consistent order for variables [x[1], x[2]]
        for var in [x[1], x[2]]:
            # Get the coefficient of the master variable in the constraint, default to 0 if not present
            coeff = constraint.get(var, 0)
            coeff_row.append(coeff)
        T_tmp.append(coeff_row)
    T[s] = T_tmp


print("Master Problem Variables (x):", [x[1], x[2]])
print("\nh vectors:")
print(h)
print("\nT matrices:")
print(T)

Master Problem Variables (x): [x_1, x_2]

h vectors:
{1: [0, 0, 6.4, 4.0, 8, 5], 2: [0, 0, 4.800000000000001, 7.2, 6, 9], 3: [0, 0, 4.800000000000001, 4.800000000000001, 6, 6], 4: [0, 0, 6.4, 8.8, 8, 11], 5: [0, 0, 3.2, 6.4, 4, 8], 6: [0, 0, 3.2, 7.2, 4, 9], 7: [0, 0, 4.800000000000001, 4.0, 6, 5], 8: [0, 0, 7.2, 7.2, 9, 9], 9: [0, 0, 3.2, 5.6000000000000005, 4, 7], 10: [0, 0, 4.800000000000001, 2.4000000000000004, 6, 3], 11: [0, 0, 7.2, 5.6000000000000005, 9, 7], 12: [0, 0, 2.4000000000000004, 4.800000000000001, 3, 6], 13: [0, 0, 1.6, 8.8, 2, 11], 14: [0, 0, 5.6000000000000005, 8.0, 7, 10], 15: [0, 0, 1.6, 8.8, 2, 11], 16: [0, 0, 7.2, 3.2, 9, 4], 17: [0, 0, 4.0, 6.4, 5, 8], 18: [0, 0, 4.0, 8.0, 5, 10], 19: [0, 0, 3.2, 4.800000000000001, 4, 6], 20: [0, 0, 3.2, 6.4, 4, 8], 21: [0, 0, 4.800000000000001, 8.0, 6, 10], 22: [0, 0, 6.4, 2.4000000000000004, 8, 3], 23: [0, 0, 4.0, 8.0, 5, 10], 24: [0, 0, 4.0, 2.4000000000000004, 5, 3], 25: [0, 0, 2.4000000000000004, 8.0, 3, 10], 26: [0, 0, 2.40

In [8]:
STOP_COND_1 = False
STOP_COND_2 = False
STOP_COND_3 = False
FEASIBILITY = False
OPTIMALITY = False


while not STOP_COND_3:
    STOP_COND_2 = False
    STOP_COND_1 = False

    while not STOP_COND_2:
        master.solve(PULP_CBC_CMD(msg=False))
        print(f" x1 is : {x[1].varValue} , x2 is : {x[2].varValue}")
        print("\n")
        STOP_COND_1 = True

        FEASIBILITY = check_cut_feasibility(x, master, h, T, scenario)    
        print(f"\nRESULT OF FEASIBILITY: {FEASIBILITY}")
        
        if FEASIBILITY:
            STOP_COND_2 = True
        else:
            print("ADDIN FEASIBILITY CUT ...")
            STOP_COND_2 = False
            STOP_COND_1 = False
        print("\n", '***' * 15, "\n")

    OPTIMALITY = check_cut_optimality(x, master, h, T, scenario, theta)
    print(f"\nRESULT OF OPTIMALITY: {OPTIMALITY}")
    
    if OPTIMALITY:
        STOP_COND_3 = True
    else:
        print("ADDIN OPTIMALITY CUT ...")
        STOP_COND_3 = False
        STOP_COND_2 = False
        STOP_COND_1 = False
    print("\n", '***' * 15, "\n")

print(f"FINAL SOLUTION: X1: {x[1].varValue}, X2: {x[2].varValue}")

 x1 is : 0.0 , x2 is : 0.0


added feasibility cut to master problem :   0.27272727*x_1 + 0.090909091*x_2 >= 10.4
added feasibility cut to master problem :   0.27272727*x_1 + 0.090909091*x_2 >= 12.0
added feasibility cut to master problem :   0.27272727*x_1 + 0.090909091*x_2 >= 9.600000000000001
added feasibility cut to master problem :   0.27272727*x_1 + 0.090909091*x_2 >= 15.200000000000001
added feasibility cut to master problem :   0.27272727*x_1 + 0.090909091*x_2 >= 9.600000000000001
added feasibility cut to master problem :   0.27272727*x_1 + 0.090909091*x_2 >= 10.4
added feasibility cut to master problem :   0.27272727*x_1 + 0.090909091*x_2 >= 8.8
added feasibility cut to master problem :   0.27272727*x_1 + 0.090909091*x_2 >= 14.4
added feasibility cut to master problem :   0.27272727*x_1 + 0.090909091*x_2 >= 8.8
added feasibility cut to master problem :   0.27272727*x_1 + 0.090909091*x_2 >= 7.200000000000001
added feasibility cut to master problem :   0.27272727*x_1 + 0.0909090